# Maritime IR-Visible Görüntü Füzyonu — Düzeltilmiş Uygulama v4

Bu notebook, **"Infrared and visible image fusion for shipborne electro-optical pod in maritime environment"**
(Liu, Dong, Xu — Infrared Physics & Technology 128 (2023) 104526) makalesindeki Eq. (1)–(22) adımlarını uygular.

**Önemli not:** Makalenin yazarları kodlarını herhangi bir yerde (GitHub, ek materyal vb.) paylaşmamıştır.
Makalede yalnızca bir **veri seti** paylaşılmıştır (`https://github.com/YaoChen8848/Maritime-infrared-and-visibke-image-data-set`),
kod paylaşılmamıştır. Bu yüzden bütün pipeline makale metnindeki formüllerden yeniden inşa edilmiştir.

## Önceki sürümde (v3) bulunan asıl hata

Önceki notebook'ta final görüntünün beklenildiği gibi çıkmamasının (yani RGB'nin renk bilgisi + IR'nin
arka plan/hedef bilgisinin görsel olarak birleşmemesinin) **kök nedeni** şu:

- Eq. (14)-(17)'de renk alt-uzayı (`ℜ`, `o_R`, `o_G`, `o_B`) taşmayı önlemek için RGB `[0, 1]` aralığına
  normalize edilerek hesaplanmıştı. Bu adım kendi içinde doğru bir mühendislik kararı (PCA/kovaryans
  sayısal olarak daha kararlı çalışır).
- **Ancak** bu `[0, 1]` ölçeğindeki `ℜ`, `o_R`, `o_G`, `o_B` değerleri hiçbir zaman geri `0-255` ölçeğine
  taşınmadan doğrudan Eq. (18) ve Eq. (19)'da, `0-255` ölçeğindeki `ID` (kızılötesi degradasyon görüntüsü)
  ile toplanıyordu.
- Sonuç: `IV = 0.2 * ID + 0.8 * (ID_ortalama − ℜ_ortalama)` işleminde `ℜ_ortalama ≈ 0.5` gibi ihmal edilebilir
  kalırken, renk katkısı `o_R, o_G, o_B` (yaklaşık `±0.3` mertebesinde) `IV`'ye (0-255 mertebesinde) eklendiğinde
  **görsel olarak yok denecek kadar küçük** kalıyordu. Yani RGB'den gelen renk bilgisi pratikte füzyon
  görüntüsüne hemen hiç yansımıyordu — tam da sizin fark ettiğiniz sorun.

## Bu sürümdeki düzeltme

Renk alt-uzayı hesaplaması **sayısal kararlılık için** yine normalize `[0, 1]` RGB üzerinde yapılıyor
(makalenin orijinal formülasyonunda böyle bir normalize adımı yok, ama ham `0-255` RGB ile kovaryans/özdeğer
ayrışımı bazı görüntülerde `ℜ`'yi taşırabiliyor). **Farkı şu:** `Re_img`, `o_R`, `o_G`, `o_B` hesaplandıktan
hemen sonra **`× 255` ile tekrar `0-255` ölçeğine geri ölçekleniyor**, böylece Eq. (18) ve Eq. (19)'daki
toplamalar artık aynı fiziksel/görsel ölçekte gerçekleşiyor. Bu, makalenin `Eq. (14)-(19)` formülasyonuna
sadık kalırken taşma hatasını da önleyen doğru düzeltmedir.

Aşağıda her fonksiyon, ilgili denklem numarasıyla birlikte satır satır yorumlanmıştır.

---

## v5 GÜNCELLEMESİ — ikinci bir taşma hatası düzeltildi

v4'te `Re, o_R, o_G, o_B` değerleri `×255` ile `0-255` ölçeğine geri taşınmıştı, ancak bu, PCA'dan gelen
**L2-normalize (birim uzunluklu) `φ` vektörü** ile doğrudan izdüşüm alındığında parlak piksellerde
`pixel·φ` değerinin `√3 ≈ 1.73`'e kadar çıkabilmesi nedeniyle **yeni bir taşma hatasına** yol açtı
(`Re` gerçek görüntülerde 400+ değerlere ulaşıyor, `IV` tamamen negatif çıkıyor, füzyon görüntüsü
neredeyse siyaha düşüyor).

**v5 düzeltmesi:** `φ` PCA yönü için L2-normalize hesaplanmaya devam ediyor, ama izdüşüm alınmadan hemen
önce **L1-normalize** ediliyor (bileşenler toplamı = 1). Böylece `ℜ` gerçek bir ağırlıklı R/G/B ortalaması
gibi davranıyor, `[0,1]` aralığında kalıyor ve `×255` sonrası `ID` ile aynı, tutarlı ölçekte oluyor.
Ayrıca güvenlik için `Re` `[0,1]`'e kırpılıyor. İlgili fonksiyon ve açıklaması Bölüm 5'te güncellenmiştir.

## 0. Kurulum

In [ ]:


!pip install -q opencv-python matplotlib numpy


import os


import numpy as np


import cv2


import matplotlib.pyplot as plt


print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)


## 1. Kızılötesi Arka Plan Yeniden Oluşturma — Eq. (1)–(7)

Makalenin fikri: Denizcilik ortamında IR görüntüsü gökyüzü + deniz yüzeyinden oluşan pürüzsüz bir arka plana
sahiptir. Bu arka planı, görüntünün satır (dikey) yönündeki ortalama parlaklık profiline **sigmoid** (lojistik)
bir eğri oturtarak modelliyoruz (Eq. 6). Sigmoid'in iki ucu, gökyüzünün atmosferik öz-ışınımına (`ω`, Eq. 4)
ve deniz yüzeyinin öz-ışınımına (`h`, Eq. 5) karşılık gelir; geçiş noktası ise ufuk çizgisinin konumudur (`α`).

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (1)-(7) — IR görüntüsündeki deniz+gökyüzü arka planını
# (nesnelerden bağımsız, pürüzsüz bir B(i,j) görüntüsü olarak) yeniden oluşturur.
# Bu fonksiyon, pipeline'ın ilk adımıdır ve daha sonra hedef (gemi/tekne)
# bölgelerini arka plandan ayırt edebilmek için bir referans oluşturur.
# ============================================================

def reconstruct_ir_background(ir_img, T=2, k=0.5):
    """
    Kızılötesi (IR) görüntü içerisindeki deniz-gökyüzü arka planını
    matematiksel bir model kullanarak yeniden oluşturur.

    Temel fikir:
    IR görüntüsünde gökyüzü ve deniz bölgesi tamamen rastgele değildir.
    Dikey yönde (yukarıdan aşağıya doğru) sıcaklık ve ışınım değişimi
    daha düzenli bir yapı gösterir.

    Bu nedenle:
    1) Görüntünün her satırındaki ortalama parlaklık bulunur.
    2) Gökyüzü ve deniz bölgelerinin ortalama ışınım seviyeleri hesaplanır.
    3) Ufuk çizgisi belirlenir.
    4) Gökyüzünden denize geçiş sigmoid fonksiyonu ile modellenir.
    5) Elde edilen tek boyutlu profil tüm görüntü genişliğine yayılır.

    Böylece nesnelerden (gemi, tekne vb.) bağımsız,
    yalnızca doğal arka planı temsil eden B görüntüsü elde edilir.

    Parametreler:
        ir_img : 2D (grayscale) IR görüntüsü (numpy array)
        T      : Eq.(4)-(5)'te atmosfer/deniz öz-ışınımını tahmin etmek için
                  kullanılan, görüntünün en üst/en alt kaç satırının ortalamasının
                  alınacağını belirten parametre (varsayılan 2 satır)
        k      : Eq.(6)'daki sigmoid fonksiyonunun eğimini (ne kadar "keskin" geçiş
                  yapacağını) kontrol eden katsayı

    Dönüş:
        background : ir_img ile aynı boyutlarda, yalnızca dikey konuma bağlı
                      pürüzsüz bir arka plan görüntüsü (0-255 aralığına kırpılmış)
    """

    # Görüntüyü float64'e çeviriyoruz ki ondalıklı ara işlemlerde taşma/yuvarlama hatası olmasın.
    I = np.asarray(ir_img, dtype=np.float64)

    # Fonksiyon yalnızca tek kanallı (grayscale) IR görüntüsü ile çalışabilir; 3 kanallı
    # (renkli) bir görüntü verilirse hatalı sonuç üretmemek için burada durduruyoruz.
    if I.ndim != 2:
        raise ValueError("IR görüntü grayscale (2D) olmalıdır.")

    M, N = I.shape  # M: satır sayısı (dikey/yükseklik), N: sütun sayısı (yatay/genişlik)

    # --- Eq. (2): Gv(j) — dikey yöndeki gri seviye izdüşümü ---
    # Her satırın (yükseklikteki her konumun) tüm sütunlar boyunca ortalama parlaklığı.
    # Bu, gökyüzünden denize doğru pürüzsüz geçen parlaklık profilini verir (bkz. Fig.3c).
    # axis=1 -> her satır için sütunlar boyunca (yatay eksende) ortalama al, sonuç (M,) boyutunda bir vektör.
    Gv = I.mean(axis=1)  # shape: (M,)

    # T, görüntü yüksekliğinin yarısını aşamaz (küçük görüntülerde güvenlik).
    # min(T, M//2): T'nin görüntü yüksekliğinin yarısından büyük olmasını engeller.
    # max(1, ...): T'nin en az 1 olmasını garanti eder (0 satırlık ortalama anlamsız olurdu).
    T = int(max(1, min(T, M // 2)))

    # --- Eq. (4): ω — atmosferin öz-ışınımı ---
    # Deniz yüzeyinden en uzak (görüntünün en üst) T satırının ortalaması alınır;
    # bu bölge atmosferin kendi ışınımına en yakın olan bölgedir.
    # Gv[:T] -> ilk T satırın (görüntünün üst kısmı, tipik olarak gökyüzü) ortalama parlaklık değerleri.
    omega = Gv[:T].mean()

    # --- Eq. (5): h — deniz yüzeyinin öz-ışınımı ---
    # Sensöre en yakın (görüntünün en alt) T satırının ortalamasından atmosfer ışınımı çıkarılır.
    # Gv[-T:] -> son T satırın (görüntünün alt kısmı, tipik olarak deniz) ortalama parlaklık değerleri.
    # omega çıkarılarak yalnızca "deniz kaynaklı ek ışınım" izole edilir.
    h = Gv[-T:].mean() - omega

    # --- Ufuk konumu α ---
    # Fig.3(b)'de belirtildiği gibi, ufuk bandı atmosferik yol ışınımının en yoğun olduğu, dolayısıyla
    # en parlak bölgedir. Bu yüzden α, Gv'nin maksimum olduğu satır indeksi olarak alınır.
    # np.argmax(Gv) -> Gv vektöründeki en büyük değerin bulunduğu indeksi (satır numarasını) döndürür.
    alpha = int(np.argmax(Gv))

    # --- Eq. (6): F(j) — sigmoid (lojistik) arka plan modeli ---
    # j (satır indeksi) α'dan çok küçükse (gökyüzü bölgesi) F(j) -> ω
    # j, α'dan çok büyükse (deniz bölgesi)          F(j) -> h + ω
    # Üstel terimi [-60, 60] ile kırpıyoruz ki exp() taşması (overflow) oluşmasın.
    # j: 0'dan M-1'e kadar tüm satır indekslerini içeren bir vektör (her satır için hesap yapılacak).
    j = np.arange(M, dtype=np.float64)
    # exponent: sigmoid'in üs kısmı, k*(alpha-j). np.clip ile [-60,60] aralığına sınırlanır çünkü
    # exp(çok büyük sayı) sayısal taşmaya (inf) yol açabilir; exp(-60) ve exp(60) zaten pratikte
    # 0'a veya çok büyük bir sayıya yakınsadığından kırpma sonucu bozmaz.
    exponent = np.clip(k * (alpha - j), -60.0, 60.0)
    # Sigmoid formülü: F(j) = h / (1 + e^exponent) + omega.
    # j << alpha (gökyüzü) iken exponent büyük pozitif -> e^exponent büyük -> h/(...) -> 0 -> F(j)=omega.
    # j >> alpha (deniz) iken exponent büyük negatif -> e^exponent -> 0 -> h/(1+0)=h -> F(j)=h+omega.
    F = h / (1.0 + np.exp(exponent)) + omega

    # --- Eq. (7): B(i,j) = F(j) ---
    # Arka plan yalnızca satır (dikey) konumuna bağlıdır; her satırdaki tüm sütunlarda aynı değeri alır.
    # F[:, None] -> F vektörünü (M,) boyutundan (M,1) boyutuna genişletir (sütun vektörü yapar).
    # np.repeat(..., N, axis=1) -> bu sütunu N kez yan yana tekrarlayarak (M,N) boyutunda,
    # her satırda sabit değerli bir görüntü üretir (yatayda değişmeyen, dikeyde sigmoid profilli).
    background = np.repeat(F[:, None], N, axis=1)

    # Olası küçük sayısal taşmaları (örn. -0.0001 veya 255.0001 gibi) [0,255] aralığına kırpar;
    # görüntü olarak görüntülenebilir/işlenebilir olmasını garanti eder.
    return np.clip(background, 0, 255)


## 2. Düşük Çözünürlüklü Görüntü — Eq. (8)

Görüntü `d x d` boyutlu bloklara ayrılır, her blok içinde Gauss ağırlıklı ortalama alınarak (blok merkezine
yakın pikseller daha fazla ağırlık kazanır) düşük çözünürlüklü bir temsil elde edilir. Bu, görüntünün
"arka plan + gürültü" bileşenini temsil eder; hedefler bu düşük çözünürlüklü temsilde kaybolur.

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (8) — Görüntünün blok tabanlı, Gauss ağırlıklı düşük
# çözünürlüklü temsilini (R) hesaplar. Bu, sonraki adımda (Eq. 9) ham görüntüden
# çıkarılarak yüksek frekanslı (hedef içerebilecek) bileşeni ortaya çıkarmakta kullanılır.
# ============================================================

def low_resolution_image(ir_img, d=16):
    """
    Eq. (8): Blok tabanlı, Gauss ağırlıklı düşük çözünürlüklü temsil R(i,j).

    d, 2'nin kuvveti olmalıdır (4, 8, 16, 32, ...) çünkü görüntü blok blok işlenir.

    Parametreler:
        ir_img : 2D IR görüntüsü
        d      : blok kenar uzunluğu (görüntü dxd boyutlu bloklara ayrılır)

    Dönüş:
        R : ir_img ile aynı boyutlarda, düşük çözünürlüklü (pürüzsüzleştirilmiş) görüntü
    """

    # d'nin 2'nin kuvveti olup olmadığını bit-düzeyinde kontrol eder: (d & (d-1)) == 0
    # ifadesi yalnızca d bir "2'nin kuvveti" (1,2,4,8,16,...) olduğunda True döner.
    # d < 2 durumunu da ayrıca reddediyoruz çünkü d=1 anlamsız bir blok boyutudur.
    if d < 2 or (d & (d - 1)) != 0:
        raise ValueError("d değeri 2^n olmalıdır. Örn: 4, 8, 16, 32.")

    # Ondalıklı hassasiyetle işlem yapabilmek için float64'e çeviriyoruz.
    I = np.asarray(ir_img, dtype=np.float64)
    H, W = I.shape  # H: yükseklik (satır sayısı), W: genişlik (sütun sayısı)

    # Görüntüyü d'ye tam bölünecek şekilde kırpıyoruz (blok toplama işlemi için gerekli).
    # Tamsayı bölme (//) ile en büyük d-katı boyutları bulunur; kalan piksel şeridi atılır.
    h2 = (H // d) * d
    w2 = (W // d) * d

    # Eğer görüntü d'den küçükse hiç tam blok oluşturulamaz; bu durumda anlamlı bir
    # düşük-çözünürlük görüntüsü üretilemeyeceği için hata fırlatılır.
    if h2 == 0 or w2 == 0:
        raise ValueError("Görüntü boyutu d değerinden küçük.")

    # Görüntüyü [0:h2, 0:w2] aralığına kırpar; böylece kalan kısım tam sayıda dxd bloğa bölünebilir.
    cropped = I[:h2, :w2]

    # Eq. (8)'deki G(m,n) Gauss çekirdeği: blok merkezine yakın pikseller daha ağırlıklı.
    # sigma: Gauss dağılımının standart sapması; d/3 kuralı, çekirdeğin d genişliğine
    # makul şekilde "sığmasını" sağlayan yaygın bir kural (1e-6 ile sıfıra bölünme engellenir).
    sigma = max(d / 3.0, 1e-6)
    # cv2.getGaussianKernel(d, sigma) -> uzunluğu d olan, 1 boyutlu (dikey) Gauss ağırlık vektörü üretir.
    g = cv2.getGaussianKernel(d, sigma)
    # g @ g.T -> 1D dikey Gauss vektörünü kendisiyle dış çarpıma sokarak 2D (d x d) simetrik
    # bir Gauss çekirdeği (merkeze yakın piksellere yüksek ağırlık veren) elde eder.
    G = g @ g.T
    # Ağırlıkların toplamını 1'e normalize ediyoruz; böylece G ile ağırlıklı toplam alındığında
    # sonuç, orijinal piksel değerlerinin "ağırlıklı ortalaması" olur (ölçek kaymaz).
    G /= G.sum()  # ağırlıklar toplamı 1 olsun (ortalama koruma)

    # Görüntüyü (satır_blok, d, sütun_blok, d) şeklinde yeniden şekillendirip
    # her blok içinde Gauss ağırlıklı toplam alıyoruz -> küçük (low-res) görüntü.
    # reshape(h2//d, d, w2//d, d): görüntüyü, her biri dxd boyutunda olan
    # (h2/d) x (w2/d) adet bloğa ayıran 4 boyutlu bir array'e dönüştürür.
    blocks = cropped.reshape(h2 // d, d, w2 // d, d)
    # G[None, :, None, :]: G çekirdeğini blocks ile aynı 4 boyutlu şekle yayınlanabilir (broadcast
    # edilebilir) hale getirir. blocks * G ile her bloğun her pikseli kendi Gauss ağırlığıyla çarpılır;
    # np.sum(..., axis=(1,3)) ile her blok içindeki (d x d) ağırlıklı pikseller toplanarak
    # blok başına TEK bir değere indirgenir -> küçük (h2/d, w2/d) boyutlu bir görüntü elde edilir.
    small = np.sum(blocks * G[None, :, None, :], axis=(1, 3))

    # Küçük görüntüyü orijinal boyuta geri büyütüyoruz (kübik interpolasyon ile) — R(i,j).
    # cv2.resize(small, (W, H), ...) -> hedef boyut (genişlik, yükseklik) sırasıyla verilir (OpenCV kuralı).
    # INTER_CUBIC: komşu pikseller arasında yumuşak/pürüzsüz bir enterpolasyon sağlar.
    R = cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC)

    return R


## 3. Kızılötesi Hedef Çıkarımı — Eq. (9)–(12)

- Eq. (9): Ham IR görüntüsünden düşük çözünürlüklü (arka plan) bileşen çıkarılır → yüksek frekanslı özellik haritası `FM`.
- Eq. (10)-(11): Deniz çırpıntısını (sea clutter) bastırmak için "merkez ağırlıklı" bir konvolüsyon çekirdeği
  uygulanır. Bu çekirdek, komşuluğunda benzer parlaklıkta piksel bulunan noktaları (periyodik dalga çırpıntısı)
  bastırırken, komşuluğunda **tekil/benzersiz** olan noktaları (gerçek hedefler) korur.
- Eq. (12): Kalan özellik haritası `[0,1]`'e normalize edilip `γ` kuvvetiyle güçlendirilir (kontrastı artırır).

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (9)-(12) — IR görüntüsündeki potansiyel hedefleri (gemi/tekne vb.)
# düşük çözünürlüklü arka plandan ayırt eder, deniz çırpıntısını (clutter) bastırır ve
# kontrastı gamma düzeltmesiyle güçlendirir.
# ============================================================

def extract_ir_targets(ir_img, d=16, X=32, gamma=2.0):
    """
    Eq. (9)-(12): Kızılötesi hedef bölgelerini (Target) çıkarır ve kontrastını artırır.

    d     : low_resolution_image için blok boyutu
    X     : Eq. (10)'daki konvolüsyon çekirdeğinin boyutu (komşuluk büyüklüğü, makalede X=32)
    gamma : Eq. (12)'deki doğrusal olmayan güçlendirme üssü (makalede gamma=2)

    Dönüş:
        R      : düşük çözünürlüklü (arka plan+gürültü) temsil, Eq.(8)
        FM     : ham - düşük çözünürlük farkı (yüksek frekanslı bileşen), Eq.(9)
        SM     : çırpıntı-bastırılmış harita (konvolüsyon sonrası), Eq.(11)
        target : [0,255] aralığında, gamma ile kontrastı güçlendirilmiş nihai hedef haritası, Eq.(12)
    """

    # Ondalıklı hassasiyetle çalışmak için float64'e dönüştürüyoruz.
    I = np.asarray(ir_img, dtype=np.float64)

    # --- Eq. (8): düşük çözünürlüklü (arka plan+gürültü) temsil ---
    # Bir önceki hücrede tanımlanan fonksiyonu çağırarak görüntünün pürüzsüzleştirilmiş halini alıyoruz.
    R = low_resolution_image(I, d=d)

    # --- Eq. (9): FM = I ⊖ R ---
    # Ham görüntüden düşük çözünürlüklü bileşen çıkarılır; geriye yüksek frekanslı
    # (potansiyel hedef + çırpıntı) bilgi kalır.
    # I - R: her pikselde, orijinal değer ile "yerel pürüzsüz ortalama" arasındaki fark alınır;
    # bu, ani parlaklık değişimlerini (kenarlar, küçük parlak/karanlık nesneler) ortaya çıkarır.
    FM = I - R

    # --- Eq. (10): deniz çırpıntısını bastıran konvolüsyon çekirdeği ---
    # Merkez eleman +X^2, çevresindeki tüm elemanlar -1; toplam çekirdek 1/X^2 ile ölçeklenir.
    # Mantık: bir pikselin komşuluğunda (X x X) benzer parlaklıkta çok sayıda piksel varsa
    # (periyodik dalga çırpıntısı gibi), bu komşuların negatif katkısı merkezin pozitif
    # katkısını götürür -> piksel bastırılır. Ama piksel komşuluğunda TEKİLSE (gerçek hedef),
    # negatif katkılar birbirini götürmez -> piksel korunur/güçlenir.
    # np.ones((X, X)) ile başlayıp negatifini alarak, çekirdeğin tamamını -1 değerleriyle dolduruyoruz.
    kernel = -np.ones((X, X), dtype=np.float64)
    # Çekirdeğin tam ortasındaki indeksi (satır ve sütun olarak) hesaplıyoruz.
    center = X // 2
    # Merkez pikseli +X^2 yapıyoruz; böylece kernel'in toplamı (X^2 - (X^2-1)) = 1 olur
    # (ölçekleme sonrası ortalama koruyan bir yapı elde edilir).
    kernel[center, center] = X ** 2
    # Çekirdeği X^2'ye bölerek normalize ediyoruz (aşırı büyük değerler üretmesini engeller).
    kernel /= X ** 2

    # --- Eq. (11): SM = FM * Kernel (konvolüsyon) ---
    # cv2.filter2D: FM görüntüsüne yukarıda tanımlanan özel çekirdeği uygular (2D konvolüsyon/korelasyon).
    # cv2.CV_64F: çıktının float64 hassasiyetinde olmasını sağlar (taşma/kırpma olmasın diye).
    # borderType=cv2.BORDER_REFLECT: görüntü kenarlarında, kenar dışı pikselleri "aynalayarak"
    # doldurur; böylece kenarlarda yapay siyah/sıfır şeritler oluşmaz.
    SM = cv2.filter2D(FM, cv2.CV_64F, kernel, borderType=cv2.BORDER_REFLECT)

    # --- Eq. (12): doğrusal olmayan (gamma) güçlendirme ---
    # Önce [0,1]'e normalize edilir (Nor(.)), sonra gamma kuvveti alınıp 0-255'e ölçeklenir.
    # SM'nin minimum ve maksimum değerlerini buluyoruz; bunlar min-max normalizasyonu için gerekli.
    sm_min = SM.min()
    sm_max = SM.max()
    # (SM - sm_min) / (sm_max - sm_min): tüm SM değerlerini [0,1] aralığına ölçekler.
    # +1e-12: sm_max == sm_min olduğu (tamamen düz/sabit görüntü) durumda sıfıra bölünmeyi engeller.
    SM_norm = (SM - sm_min) / (sm_max - sm_min + 1e-12)
    # SM_norm ** gamma: gamma>1 olduğunda düşük değerleri daha da bastırır, yüksek (hedef
    # olma ihtimali yüksek) değerleri nispeten korur -> kontrast artışı sağlar.
    # *255.0: sonucu görüntü ölçeğine (0-255) geri taşır.
    target = (SM_norm ** gamma) * 255.0

    # Ara sonuçların tümü (R, FM, SM) teşhis/görselleştirme amaçlı döndürülür; target ise
    # olası ufak taşmaları önlemek için son kez [0,255]'e kırpılarak döndürülür.
    return R, FM, SM, np.clip(target, 0, 255)


## 4. Kızılötesi Degradasyon Görüntüsü — Eq. (13)

Arka plan (Eq. 7) ile hedef bölgesi (Eq. 12) toplanır ve `[0,255]`'e normalize edilir. Bu, "gerçekçi olmayan
ama hem pürüzsüz arka planı hem de belirgin hedefleri içeren" sentetik bir IR görüntüsüdür (`ID`).

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (13) — Arka plan (B) ile hedef haritasını (Target) toplayarak,
# hem pürüzsüz arka planı hem de belirgin hedefleri içeren "sentetik/degradasyon"
# IR görüntüsü ID'yi oluşturur. Bu görüntü, gerçek IR'den daha "temiz" ama
# yapay (degraded/idealized) bir temsildir.
# ============================================================

def build_ir_degraded_image(background, target):
    """Eq. (13): ID = Nor(B + Target) * 255"""

    # Arka plan ve hedef haritasını piksel piksel topluyoruz; float64'e çevirerek
    # olası taşma/tip uyuşmazlığı sorunlarının önüne geçiyoruz.
    combined = np.asarray(background, dtype=np.float64) + np.asarray(target, dtype=np.float64)

    # Toplam görüntünün min ve maksimum değerlerini buluyoruz (normalizasyon için gerekli).
    cmin = combined.min()
    cmax = combined.max()

    # Min-max normalizasyonu: (combined - cmin) / (cmax - cmin) ifadesi değerleri [0,1]
    # aralığına sıkıştırır, ardından *255.0 ile görüntü (0-255) ölçeğine geri taşınır.
    # +1e-12: cmax == cmin (sabit/düz görüntü) durumunda sıfıra bölünmeyi engeller.
    ID = (combined - cmin) / (cmax - cmin + 1e-12) * 255.0

    # Olası ufak sayısal taşmaları [0,255] sınırlarına kırpıp döndürüyoruz.
    return np.clip(ID, 0, 255)


## 5. Görünür Görüntüden Renk Özelliği Çıkarımı — Eq. (14)–(17)

Fikir: RGB görüntüsü, üç bileşenli (R,G,B) bir rastgele değişken olarak ele alınır. Kovaryans matrisinin en
büyük özdeğerine karşılık gelen özvektör yönünde bir izdüşüm alınarak **tek boyutlu** bir "parlaklık/gri
öz" (`ℜ`) elde edilir (bu, PCA'nın birinci temel bileşenine karşılık gelir). RGB'den bu tek boyutlu bileşen
çıkarılınca geriye **sadece renk bilgisini** taşıyan `o_R, o_G, o_B` kalır (Eq. 17).

### ⚠️ v4'te ortaya çıkan İKİNCİ hata (ve buradaki düzeltmesi)

v4'te `Re, o_R, o_G, o_B` değerlerini `0-255` ölçeğine geri taşımak (`×255`) renk bilgisinin görünür
olmasını sağladı, **ama** yeni bir taşma hatası ortaya çıkardı:

- Kovaryans/özdeğer ayrışımından çıkan `φ` (phi) bir **L2-normalize (birim uzunluklu, `‖φ‖₂=1`) vektördür**.
- Korelasyonlu (gerçek fotoğraflardaki gibi) RGB kanalları için `φ`, tipik olarak
  `(0.577, 0.577, 0.577)` civarına yakın çıkar (`1/√3` her bileşende).
- Parlak/beyaza yakın bir piksel `(1,1,1)` için `pixel · φ ≈ √3 ≈ 1.73` olur — yani **`[0,1]` aralığını
  aşar**! `×255` ile ölçeklenince bu `Re` değerlerinin `255`'i, hatta örneğimizde `437`'yi aşmasına yol
  açtı. Sonuç: `IV = (1-β)·ID + β·(ID_ortalama − ℜ_ortalama)` tamamen negatif çıktı, `IC` de negatif kaldı
  ve nihai füzyon görüntüsü neredeyse tamamen siyaha düştü.

**Düzeltme:** PCA yönünü doğru bulmak için `φ`'yi yine L2-normalize (birim vektör) olarak hesaplıyoruz,
fakat **projeksiyonu almadan hemen önce `φ`'yi L1-normalize ediyoruz** (bileşenler toplamı = 1 olacak
şekilde). Böylece `ℜ = ρ · (pixels · φ_L1)` artık gerçek bir **ağırlıklı ortalama** (R,G,B'nin ağırlıklı
grinin) gibi davranır ve girdi piksel değerleriyle **aynı `[0,1]` aralığında** kalır — taşma olmaz. Yön
(hangi renk ekseni en çok varyansı açıklıyor) hâlâ orijinal PCA'dan geliyor; sadece büyüklüğü (magnitude)
piksel ölçeğiyle tutarlı hâle getiriliyor.

Ayrıca güvenlik amacıyla, olası aşırı uç (outlier) pikseller için `Re` ve renk özellikleri `[0,1]`
aralığına kırpılıyor; bu sayede pipeline hiçbir görüntüde negatif/taşmış değerlerle karşılaşmıyor.

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (14)-(17) — Görünür (RGB) görüntüden, PCA (Temel Bileşen
# Analizi) benzeri bir yöntemle tek boyutlu bir "parlaklık/renk özü" (Re) çıkarır
# ve her kanalın bu özden ne kadar saptığını (o_R, o_G, o_B renk özellikleri) hesaplar.
# Bu değerler daha sonra IR kaynaklı yoğunlukla (IV) birleştirilerek renkli füzyon
# görüntüsü oluşturulacaktır (Eq. 19).
# ============================================================

def extract_visible_color_features(vis_bgr):
    """
    Eq. (14)-(17): Görünür (RGB) görüntüden renk alt-uzayını (Re) ve renk özelliklerini (o_R,o_G,o_B) çıkarır.

    Not: Ara hesap [0,1] ölçeğinde (sayısal kararlılık) yapılır, çıktı 0-255 ölçeğine geri ölçeklenir.

    KRİTİK DÜZELTME: Projeksiyon vektörü (phi) PCA için L2-normalize (birim uzunluklu) olarak
    hesaplanır, fakat asıl izdüşüm alınmadan önce L1-normalize edilir (bileşenler toplamı = 1).
    Aksi hâlde parlak piksellerde projeksiyon degeri sqrt(3)'e kadar (yaklasik 1.73x) sismekte ve
    Re, orijinal piksel aralığını (0-1) taşarak sonraki adımlarda (Eq.18) IV'nin tamamen negatif
    çıkmasına, dolayısıyla füzyon görüntüsünün neredeyse siyaha düşmesine yol açmaktadır.
    """

    # Ondalıklı hassasiyetle çalışabilmek için float64'e dönüştürüyoruz.
    V = np.asarray(vis_bgr, dtype=np.float64)

    # Fonksiyon yalnızca 3 kanallı (B,G,R) renkli görüntülerle çalışabilir; farklı bir
    # boyuta sahip girdi verilirse anlamlı olmayan sonuç üretmemek için hata fırlatıyoruz.
    if V.ndim != 3 or V.shape[2] != 3:
        raise ValueError("Visible görüntü 3 kanallı BGR olmalıdır.")

    H, W, _ = V.shape  # H: yükseklik, W: genişlik (kanal sayısını burada kullanmıyoruz)

    # OpenCV BGR sırasında okur; burada RGB sırasına çevirip [0,1]'e normalize ediyoruz.
    # V[:,:,2] -> Kırmızı (R) kanal (OpenCV'de indeks 2), .reshape(-1) ile tek boyutlu
    # (H*W,) uzunluğunda bir vektöre düzleştiriyoruz, /255.0 ile [0,1] aralığına ölçekliyoruz.
    R = V[:, :, 2].reshape(-1) / 255.0
    # V[:,:,1] -> Yeşil (G) kanal, aynı işlemler uygulanıyor.
    G = V[:, :, 1].reshape(-1) / 255.0
    # V[:,:,0] -> Mavi (B) kanal (OpenCV'de indeks 0), aynı işlemler uygulanıyor.
    B = V[:, :, 0].reshape(-1) / 255.0

    # np.column_stack: R, G, B vektörlerini yan yana sütun olarak birleştirip,
    # her satırı bir pikselin (R,G,B) üçlüsünü temsil eden (H*W, 3) boyutlu bir matris oluşturur.
    pixels = np.column_stack((R, G, B))  # shape: (H*W, 3)

    # --- Eq. (14): kovaryans matrisi ---
    # np.cov(pixels, rowvar=False): her SÜTUNU bir değişken (R, G, B) olarak ele alıp,
    # bu üç değişken arasındaki kovaryansı hesaplayan 3x3'lük bir matris üretir.
    # Bu matris, R-G-B kanallarının birbirleriyle ne kadar "birlikte değiştiğini" (korelasyon
    # yapısını) gösterir.
    covariance = np.cov(pixels, rowvar=False)

    # --- Eq. (15)-(16): en büyük özdeğer/özvektör ile boyut indirgeme ---
    # np.linalg.eigh: simetrik (kovaryans gibi) matrisler için özdeğer/özvektör hesabında
    # np.linalg.eig'den daha kararlı ve hızlıdır; özdeğerleri KÜÇÜKTEN BÜYÜĞE sıralı döndürür.
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)  # simetrik matris -> eigh daha kararlı
    # En büyük özdeğerin (verideki en fazla varyansı açıklayan yönün) indeksini buluyoruz.
    idx = int(np.argmax(eigenvalues))

    # En büyük özdeğeri float olarak alıyoruz (Eq.16'daki rho hesabında kullanılacak).
    lambda_max = float(eigenvalues[idx])
    # eigenvectors[:, idx]: en büyük özdeğere karşılık gelen özvektör sütununu seçiyoruz;
    # .copy() ile orijinal diziden bağımsız bir kopya alıyoruz (sonradan işaretini değiştireceğiz).
    phi = eigenvectors[:, idx].copy()  # en büyük özdeğere karşılık gelen özvektör (L2-normalize, ||phi||=1)

    # Eq. (16): rho = en büyük özdeğer / tüm özdeğerlerin toplamı (birinci bileşenin
    # açıkladığı varyans oranı)
    # rho, [0,1] aralığında bir orandır; 1'e ne kadar yakınsa, tek bir yön (phi) verideki
    # varyansın o kadar büyük bir kısmını açıklıyor demektir (renk kanalları o kadar korelasyonlu).
    rho = lambda_max / (np.sum(eigenvalues) + 1e-12)

    # Özvektörün işareti keyfi olabilir (kovaryans özvektörleri +/- yönde eşdeğerdir).
    # Farklı çalıştırmalarda tutarlı sonuç almak için işareti sabitliyoruz (bileşenler toplamı pozitif olsun).
    # phi bileşenlerinin toplamı negatifse, tüm vektörü -1 ile çarparak işaretini çeviriyoruz;
    # bu, sonucu değiştirmez (aynı doğrultu) ama tutarlı bir kural (pozitif toplam) sağlar.
    if np.sum(phi) < 0:
        phi = -phi

    # === DÜZELTME: phi'yi L1-normalize et (bileşenler toplamı = 1) ===
    # phi, PCA yönünü bulmak için L2-normalize (birim uzunluklu) olarak geldi; ama izdüşümü
    # doğrudan bu haliyle almak, korelasyonlu RGB kanallarında pixel@phi değerini sqrt(3)'e kadar
    # büyütebiliyor (bkz. yukarıdaki markdown açıklaması). L1-normalize edilmiş phi_l1 ile izdüşüm
    # alındığında sonuç, R/G/B'nin AĞIRLIKLI ORTALAMASI gibi davranır ve [0,1] aralığında kalır.
    # phi'nin üç bileşeninin toplamını hesaplıyoruz.
    phi_sum = np.sum(phi)
    if abs(phi_sum) < 1e-8:
        # Dejenere durum (bileşenler toplamı ~0): güvenli varsayılan olarak eşit ağırlık kullan.
        # Bu durumda R, G, B kanallarına eşit (1/3) ağırlık verilerek basit bir gri-ton ortalaması alınır.
        phi_l1 = np.array([1.0, 1.0, 1.0]) / 3.0
    else:
        # phi'nin her bileşenini toplamına bölerek, yeni bileşenlerin toplamının tam olarak 1
        # olmasını sağlıyoruz (L1-normalizasyon); böylece izdüşüm bir "ağırlıklı ortalama" olur.
        phi_l1 = phi / phi_sum

    # --- Eq. (15): tek boyutlu stokastik değişken ℜ (artık [0,1]'de kalacak şekilde ölçeklendirilmiş) ---
    # pixels @ phi_l1: her pikselin (R,G,B) üçlüsü ile phi_l1 ağırlık vektörünün iç çarpımı
    # (matris çarpımı) alınır; bu, her piksel için TEK bir sayı (ağırlıklı parlaklık) üretir.
    # rho ile çarpılarak, bu bileşenin verideki varyansı ne kadar iyi temsil ettiği de hesaba katılır.
    Re = rho * (pixels @ phi_l1)          # shape: (H*W,)
    # Olası küçük sayısal sapmaları [0,1] aralığına kırpıyoruz (güvenlik amaçlı).
    Re = np.clip(Re, 0.0, 1.0)            # güvenlik: olası aşırı uç değerleri kırp
    # Düzleştirilmiş (H*W,) vektörü tekrar orijinal görüntü boyutuna (H,W) geri şekillendiriyoruz.
    Re_img = Re.reshape(H, W)

    # --- Eq. (17): renk özellikleri (RGB'den ℜ'nin çıkarılmasıyla elde edilir) ---
    # Her kanaldan (R,G,B) ortak "parlaklık özü" (Re_img) çıkarılarak, o kanala özgü
    # "artık renk bilgisi" (o_R, o_G, o_B) elde edilir. Bu artıklar, füzyon görüntüsüne
    # rengi geri kazandırmak için kullanılacaktır (bkz. Eq.19).
    o_R = R.reshape(H, W) - Re_img
    o_G = G.reshape(H, W) - Re_img
    o_B = B.reshape(H, W) - Re_img

    # 0-255 ölçeğine geri taşı: Eq. (18)-(19)'da bu değerler ID (0-255 ölçekli) ile toplanacak.
    # Buraya kadar tüm hesap [0,1] aralığında yürütüldü; artık bunları görüntü (0-255) ölçeğine
    # geri çeviriyoruz ki sonraki adımlarda IR tabanlı ID (0-255) ile tutarlı şekilde toplanabilsinler.
    Re_img = Re_img * 255.0
    o_R = o_R * 255.0
    o_G = o_G * 255.0
    o_B = o_B * 255.0

    return Re_img, o_R, o_G, o_B


## 6. Yoğunluk Haritası — Eq. (18)

`IV`, `ID`'nin (kızılötesi degradasyon görüntüsü) `(1-β)` ağırlıklı hâli ile, `ID` ve `ℜ`'nin **ortalamaları**
arasındaki farkın `β` ağırlıklı hâlinin toplamıdır. Makalede bu terim, gökyüzü bölgesinin parlaklığını
görünür görüntüye benzer şekilde ayarlamak için eklenmiştir (β=0.8 seçilmiştir).

### ⚠️ Üçüncü sorun ve buradaki düzeltme

`ℜ` düzeltildikten sonra bile (taşma olmadan), `ID` ve `ℜ`'nin **global ortalamaları çoğu gerçek görüntüde
çok farklı ölçeklerde** çıkabiliyor. Sebep: Eq. (13)'teki `Nor()` (min-max) normalizasyonu, arka planı
genellikle çok düşük değerlere sıkıştırıyor (çünkü sadece küçük/parlak hedef bölgesi üst sınıra ulaşıyor),
bu yüzden `ID`'nin ortalaması genelde düşük kalıyor. `ℜ` ise görünür görüntünün genel parlaklığını temsil
ettiği için çoğu zaman belirgin biçimde daha yüksek. Sonuç: `β·(ID_ortalama − ℜ_ortalama)` terimi (β=0.8
ile baskın ağırlıkta) **büyük ve negatif** çıkıyor, bu da `IV`'yi tamamen negatife itip füzyon görüntüsünü
neredeyse siyaha düşürüyor — hem IR yapısı hem de renk kayboluyor.

Makalenin diğer tüm ara adımları (arka plan, hedef, `ID`) hep `Nor()` ile `[0,255]` aralığına
normalize edilirken, Eq. (18) metninde `IV` için böyle bir normalizasyon **yazılı olarak belirtilmemiş** —
muhtemelen makalenin kendi test görüntülerinde `ID` ve `ℜ` ortalamaları birbirine yakın kaldığı için bu
adıma gerek duyulmamış. Farklı/gerçek görüntülerle sağlam çalışsın diye, pipeline'ın kendi tutarlılığına
uygun olarak, Eq. (18) çıktısına da diğer adımlarla aynı mantıkla bir `Nor()` (min-max → 0-255) uyguluyoruz.
Bu, β'nin göreli ağırlıklandırma etkisini (yani `ID`'nin uzamsal yapısı ile ortalama fark düzeltmesi
arasındaki oranı) korurken, `IV`'yi görüntülenebilir/pozitif bir aralığa taşıyor.

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (18) — IR degradasyon görüntüsü (ID) ile görünür görüntüden
# çıkarılan renk özü (Re) birleştirilerek "yoğunluk haritası" (IV) hesaplanır.
# Bu harita, IR'nin uzamsal yapısını korurken genel parlaklık seviyesini görünür
# görüntüye yaklaştırır; sonraki adımda (Eq.19) renk bilgisiyle toplanacaktır.
# ============================================================

def build_intensity_map(ID, Re, beta=0.8):
    """
    Eq. (18): IV = (1-beta)*ID + beta*(mean(ID) - mean(Re))

    DUZELTME: ID ve Re'nin global ortalamalari gercek goruntulerde genelde cok farkli
    olceklerde cikiyor (ID, Eq.13'teki agresif min-max normalizasyonu yuzunden dusuk
    ortalamaya sikismis oluyor; Re ise gorunur goruntunun tipik parlakligini tasiyor).
    Bu da beta*(ID_ort - Re_ort) teriminin (beta=0.8 ile baskin) IV'yi tamamen negatife
    itmesine yol aciyor. Pipeline'in diger tum ara adimlari (arka plan, hedef, ID) Nor()
    ile [0,255]'e normalize edildigi icin, tutarlilik ve sayisal saglamlik amaciyla IV'ye
    de ayni mantikla bir Nor() uyguluyoruz (makale metninde acikca yazilmasa da, mevcut
    pipeline'in kendi normalizasyon deseniyle uyumlu, gerekli bir muhendislik adimidir).
    """

    # Her iki girdiyi de float64'e çevirerek tutarlı, hassas aritmetik işlem sağlıyoruz.
    ID = np.asarray(ID, dtype=np.float64)
    Re = np.asarray(Re, dtype=np.float64)

    # ID görüntüsünün tüm piksellerinin ortalamasını (skaler bir sayı) hesaplıyoruz.
    ID_mean = ID.mean()
    # Re görüntüsünün tüm piksellerinin ortalamasını (skaler bir sayı) hesaplıyoruz.
    Re_mean = Re.mean()

    # (1-beta) katsayisi ID'nin uzamsal yapisini (arka plan + hedef) korur.
    # beta katsayisi, ID ve Re'nin skaler ortalamalari arasindaki farki ekleyerek
    # gokyuzu/genel parlaklik seviyesini gorunur goruntuye yaklastirir.
    # (1-beta)*ID: ID'nin piksel piksel uzamsal detayının (arka plan+hedef desenlerinin) büyük kısmını korur.
    # beta*(ID_mean - Re_mean): TÜM görüntüye eklenen SABİT bir kaydırma terimi (skaler fark).
    IV_raw = (1.0 - beta) * ID + beta * (ID_mean - Re_mean)

    # === Guvenlik/tutarlilik normalizasyonu ===
    # IV_raw negatif/asiri kaymis olabilir (yukaridaki aciklamaya bakiniz). Diger tum ara
    # adimlarla (background, target, ID) ayni Nor()->0-255 desenini uyguluyoruz.
    # IV_raw'ın en küçük ve en büyük değerlerini buluyoruz (yeniden ölçekleme için gerekli).
    iv_min = IV_raw.min()
    iv_max = IV_raw.max()

    if iv_max - iv_min < 1e-8:
        # Dejenere durum: IV_raw tamamen sabit -> orta gri deger ata.
        # Eğer görüntüdeki tüm değerler birbirine eşitse (varyans yoksa), normalizasyon
        # sıfıra bölme hatası verir; bu durumda anlamlı bir varsayılan olarak orta gri (127.5) atanır.
        IV = np.full_like(IV_raw, 127.5)
    else:
        # Standart min-max normalizasyonu: değerleri [0,1] aralığına sıkıştırıp *255 ile
        # görüntü ölçeğine (0-255) geri taşıyoruz; böylece IV, diğer ara adımlarla tutarlı bir aralıkta olur.
        IV = (IV_raw - iv_min) / (iv_max - iv_min) * 255.0

    return IV


## 7. İlk Renkli Görüntü — Eq. (19)

Yoğunluk haritası `IV` ile her kanalın renk özelliği (`o_R, o_G, o_B`) toplanarak ilk (ham) renkli füzyon
görüntüsü `IC` elde edilir. Artık `o_R, o_G, o_B` doğru (0-255) ölçekte olduğu için bu toplam, görünür
görüntünün rengini `IV`'nin (IR kaynaklı) parlaklık/yapı bilgisine gerçekten katar.

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (19) — Yoğunluk haritası (IV) ile görünür görüntüden alınan
# renk özellikleri (o_R, o_G, o_B) toplanarak, IR kaynaklı parlaklığa sahip ancak
# görünür görüntünün rengini taşıyan "ilk renkli füzyon görüntüsü" (IC) elde edilir.
# ============================================================

def build_initial_color_image(IV, o_R, o_G, o_B):
    """Eq. (19): IC_R = IV + o_R,  IC_G = IV + o_G,  IC_B = IV + o_B"""

    # Tutarlı ondalıklı aritmetik için IV'yi float64'e çeviriyoruz.
    IV = np.asarray(IV, dtype=np.float64)

    # Her renk kanalı için, ortak yoğunluk haritasına (IV) o kanala özgü renk artığı (o_X) eklenir.
    # Böylece her kanal, aynı IR-kaynaklı parlaklık yapısını paylaşır ama farklı renk tonuna sahiptir.
    IC_R = IV + o_R
    IC_G = IV + o_G
    IC_B = IV + o_B

    # OpenCV BGR kanal sırasını kullanıyoruz.
    # np.stack ile üç ayrı 2D kanalı, üçüncü eksende (axis=2) birleştirerek (H,W,3) boyutlu
    # bir renkli görüntü oluşturuyoruz. OpenCV kuralına uymak için sıralama B, G, R şeklindedir.
    IC_bgr = np.stack([IC_B, IC_G, IC_R], axis=2)

    # Burada kırpma/uint8 dönüşümü YAPMIYORUZ; sonraki YUV adımı için değer aralığını
    # teşhis edebilmek amacıyla float olarak bırakıyoruz.
    return IC_bgr


## 8. YUV Renk Uzayında Geliştirme — Eq. (20)–(22)

Son adımda, `IC` ve orijinal görünür görüntü YUV uzayına çevrilir. `Y` (parlaklık) kanalı doğrudan `IC`'den
alınır (Eq. 20). `U` ve `V` (renk/doygunluk) kanalları ise, `IC`'nin kendi ortalama/standart sapması ile
görünür görüntünün `U`/`V` istatistikleri karşılaştırılarak yeniden ölçeklenir (Eq. 21-22) — böylece füzyon
görüntüsünün renk doygunluğu, görünür görüntününkine daha çok benzer.

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (20)-(22) — İlk renkli görüntüyü (IC) ve orijinal görünür
# görüntüyü YUV renk uzayında karşılaştırarak, parlaklığı (Y) IC'den alan ama
# renk doygunluğunu (U,V) görünür görüntüye yaklaştıran NİHAİ füzyon görüntüsünü üretir.
# ============================================================

def yuv_color_enhancement(IC_bgr, vis_bgr):
    """
    Eq. (20)-(22): IC ve görünür görüntüyü YUV uzayında karşılaştırarak nihai füzyon görüntüsünü üretir.

    OpenCV'nin cvtColor fonksiyonu float64 kabul etmediği için, dönüşümden hemen önce
    float32 / [0,1] aralığına geçici olarak taşınır. Bu yalnızca OpenCV uyumluluğu için yapılan
    bir tip dönüşümüdür; IC'nin kendisi bu noktaya kadar float olarak taşınmıştır.
    """

    # Girdileri float64'e çevirerek tutarlı bir başlangıç noktası oluşturuyoruz.
    IC = np.asarray(IC_bgr, dtype=np.float64)
    VIS = np.asarray(vis_bgr, dtype=np.float64)

    # NaN/sonsuz değerleri temizleyip [0,255] aralığına kırpıyoruz (görüntüleme/OpenCV için gerekli).
    # np.nan_to_num: olası NaN değerlerini 0'a, +sonsuz değerleri 255'e, -sonsuz değerleri 0'a çevirir
    # (önceki adımlardaki bölme işlemlerinden kaynaklanabilecek sayısal anormallikleri temizler).
    # np.clip ile ardından kesin olarak [0,255] aralığına sabitliyoruz.
    IC_255 = np.clip(np.nan_to_num(IC, nan=0.0, posinf=255.0, neginf=0.0), 0.0, 255.0)
    VIS_255 = np.clip(np.nan_to_num(VIS, nan=0.0, posinf=255.0, neginf=0.0), 0.0, 255.0)

    # OpenCV'nin cvtColor fonksiyonu float32/[0,1] veya uint8 girdi beklediği için, burada
    # değerleri 255'e bölüp float32 tipine çeviriyoruz (yalnızca OpenCV uyumluluğu amaçlı).
    IC_01 = (IC_255 / 255.0).astype(np.float32)
    VIS_01 = (VIS_255 / 255.0).astype(np.float32)

    # cv2.cvtColor(..., COLOR_BGR2YUV): BGR renk uzayından YUV renk uzayına dönüşüm yapar.
    # YUV uzayı, parlaklık (Y) bilgisini renk (U,V) bilgisinden ayırır; bu da parlaklığı bir
    # görüntüden, renk doygunluğunu başka bir görüntüden almayı mümkün kılar.
    # *255.0 ile sonucu tekrar 0-255 ölçeğine geri taşıyoruz (hesaplamaları bu ölçekte sürdürmek için).
    IC_yuv = cv2.cvtColor(IC_01, cv2.COLOR_BGR2YUV).astype(np.float64) * 255.0
    V_yuv = cv2.cvtColor(VIS_01, cv2.COLOR_BGR2YUV).astype(np.float64) * 255.0

    # --- Eq. (20): FusedY = IC_Y ---
    # Nihai görüntünün parlaklık (Y) kanalı, doğrudan IC'nin Y kanalından alınır (IR kaynaklı
    # yoğunluk yapısı korunur). IC_yuv[:,:,0] -> Y kanalı (YUV sıralamasında ilk kanal).
    FusedY = IC_yuv[:, :, 0]

    # --- Eq. (21): U kanalı ---
    # IC'nin U kanalını, kendi ortalama/standart sapması ile görünür görüntünün U istatistiklerine
    # göre yeniden ölçekleyip IC_U'ya geri ekliyoruz (renk doygunluğunu görünür görüntüye yaklaştırır).
    ICU = IC_yuv[:, :, 1]  # IC'nin U (mavi-fark krominans) kanalı
    VU = V_yuv[:, :, 1]    # Görünür görüntünün U kanalı
    ICU_bar = ICU.mean()   # IC'nin U kanalının ortalaması (global istatistik)
    ICU_std = ICU.std()    # IC'nin U kanalının standart sapması (değişkenliği)
    VU_std = VU.std()      # Görünür görüntünün U kanalının standart sapması
    # Formül, IC_U ile VIS_U arasındaki farkı (ICU - VU), IC'nin ortalama parlaklığı,
    # standart sapma oranı (ICU_std/VU_std) ve VU'ya bağlı bir sönümleme (1/(1+VU)) ile
    # ölçeklendirip yarısını (0.5) alarak orijinal ICU'ya ekler; bu, renk doygunluğunu
    # görünür görüntünün istatistiklerine doğru "iterek" düzenler.
    FusedU = ICU_bar / (1.0 + VU) * (ICU_std / (VU_std + 1e-12)) * (ICU - VU) * 0.5 + ICU

    # --- Eq. (22): V kanalı (U ile aynı mantık) ---
    ICV = IC_yuv[:, :, 2]  # IC'nin V (kırmızı-fark krominans) kanalı
    VV = V_yuv[:, :, 2]    # Görünür görüntünün V kanalı
    ICV_bar = ICV.mean()
    ICV_std = ICV.std()
    VV_std = VV.std()
    # V kanalı için U ile birebir aynı mantıkla, farklı istatistiklerle hesaplama yapılır.
    FusedV = ICV_bar / (1.0 + VV) * (ICV_std / (VV_std + 1e-12)) * (ICV - VV) * 0.5 + ICV

    # Y, U, V kanallarını tekrar (H,W,3) boyutunda tek bir görüntüde birleştiriyoruz.
    fused_yuv = np.stack([FusedY, FusedU, FusedV], axis=2)
    # Olası NaN/sonsuz değerleri temizleyip [0,255]'e kırpıyoruz, ardından uint8'e (8-bit
    # tam sayı, standart görüntü formatı) çeviriyoruz; cv2.cvtColor uint8 girdi bekler.
    fused_yuv = np.clip(np.nan_to_num(fused_yuv, nan=0.0, posinf=255.0, neginf=0.0), 0, 255).astype(np.uint8)

    # YUV -> BGR dönüşümü ile nihai, görüntülenebilir füzyon görüntüsü elde edilir.
    fused = cv2.cvtColor(fused_yuv, cv2.COLOR_YUV2BGR)

    return fused


## 9. Tam Füzyon Pipeline'ı

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Eq. (1)-(22)'nin tamamını uçtan uca çalıştıran ANA PIPELINE
# fonksiyonu. Yukarıda tanımlanan tüm adım fonksiyonlarını sırasıyla çağırır ve
# tüm ara sonuçları (görselleştirme/analiz için) bir sözlük (dict) olarak döndürür.
# ============================================================

def fuse_maritime_images_paper(
    ir_img,
    vis_img,
    T=2,
    k=0.5,
    d=16,
    X=32,
    gamma=2.0,
    beta=0.8
):
    """
    Eq. (1)-(22)'yi baştan sona uygulayan tam pipeline.

    Adımlar:
      1) IR arka plan yeniden oluşturma      (Eq. 1-7)
      2) IR hedef bölgesi çıkarımı           (Eq. 8-12)
      3) IR degradasyon görüntüsü            (Eq. 13)
      4) Görünür görüntüden renk çıkarımı    (Eq. 14-17)
      5) Yoğunluk haritası                   (Eq. 18)
      6) İlk renkli görüntü                  (Eq. 19)
      7) YUV renk geliştirme                 (Eq. 20-22)
    """

    # Girdi IR görüntüsünü numpy array'e çeviriyoruz (zaten array ise dokunmadan bırakır).
    ir = np.asarray(ir_img)
    # Eğer IR görüntüsü yanlışlıkla 3 kanallı (renkli) olarak yüklendiyse, tek kanala
    # (gri tonlama) indirgiyoruz; çünkü sonraki fonksiyonlar 2D grayscale IR bekliyor.
    if ir.ndim == 3:
        ir = cv2.cvtColor(ir, cv2.COLOR_BGR2GRAY)

    # Girdi görünür görüntüyü numpy array'e çeviriyoruz.
    vis = np.asarray(vis_img)
    # Eğer görünür görüntü yanlışlıkla tek kanallı (gri) yüklendiyse, 3 kanallı BGR'a
    # genişletiyoruz; çünkü renk çıkarım fonksiyonu 3 kanal bekliyor.
    if vis.ndim == 2:
        vis = cv2.cvtColor(vis, cv2.COLOR_GRAY2BGR)

    # Dönüşümlerden sonra hâlâ beklenen boyutta değilse (bozuk/hatalı girdi), açık bir
    # hata mesajıyla işlemi durduruyoruz (sessizce yanlış sonuç üretmek yerine).
    if ir.ndim != 2:
        raise ValueError("IR image 2D grayscale olmalı.")
    if vis.ndim != 3 or vis.shape[2] != 3:
        raise ValueError("Visible image 3-kanallı olmalı.")

    # IR görüntüsünün yükseklik (H) ve genişliğini (W) referans boyut olarak alıyoruz;
    # görünür görüntü bu boyuta göre yeniden ölçeklenecek.
    H, W = ir.shape

    # NOT: Bu sadece boyut eşitleme (resize) yapar; sensörler arası GEOMETRİK hizalama
    # (kayıt/registration) gerçekleştirmez. Makale, hizalanmış (roughly aligned) çift kullanır.
    # Eğer görünür görüntünün boyutu IR ile aynı değilse, kübik interpolasyon ile IR
    # boyutuna yeniden ölçekleniyor (iki görüntünün piksel piksel eşleşmesi için gereklidir).
    if vis.shape[:2] != (H, W):
        vis = cv2.resize(vis, (W, H), interpolation=cv2.INTER_CUBIC)

    # Aşağıda pipeline'ın 7 ana adımı, önceki hücrelerde tanımlanan fonksiyonlar
    # kullanılarak sırasıyla çalıştırılıyor; her adımın çıktısı bir sonrakine girdi olur.
    background = reconstruct_ir_background(ir, T=T, k=k)                 # Eq. 1-7
    R, FM, SM, target = extract_ir_targets(ir, d=d, X=X, gamma=gamma)    # Eq. 8-12
    ID = build_ir_degraded_image(background, target)                    # Eq. 13
    Re, o_R, o_G, o_B = extract_visible_color_features(vis)              # Eq. 14-17 (düzeltilmiş ölçek)
    IV = build_intensity_map(ID, Re, beta=beta)                         # Eq. 18
    IC = build_initial_color_image(IV, o_R, o_G, o_B)                   # Eq. 19
    fused = yuv_color_enhancement(IC, vis)                              # Eq. 20-22

    # Tüm ara ve nihai sonuçları, hem son kullanım (fused) hem de teşhis/görselleştirme
    # amaçlı (background, target, ID, Re, IV, IC vb.) erişilebilir olacak şekilde bir
    # sözlük (dict) içinde döndürüyoruz.
    return {
        "ir": ir,
        "visible": vis,
        "background": background,
        "R": R,
        "FM": FM,
        "SM": SM,
        "target": target,
        "ID": ID,
        "Re": Re,
        "IV": IV,
        "IC": IC,
        "fused": fused,
    }


## 10. Görüntüleri Yükleme

Dosyaları sol paneldeki **Dosyalar (📁)** sekmesine sürükle-bırak ile `/content/` altına yükleyin,
ardından aşağıdaki yol değişkenlerini kendi dosya adlarınıza göre güncelleyip hücreyi çalıştırın.

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: IR ve görünür (visible) görüntü dosyalarının Colab ortamına
# yüklenip yüklenmediğini kontrol eder; kullanıcıyı yönlendiren bir uyarı/onay
# mesajı basar. Henüz hiçbir görüntü OKUMAZ, yalnızca dosya YOLU kontrolü yapar.
# ============================================================

# IR görüntüsünün Colab'daki (yüklenmesi beklenen) dosya yolu.
ir_path = '/content/ir.jpg'
# Görünür (visible/RGB) görüntünün Colab'daki dosya yolu.
vis_path = '/content/vis.jpg'

# os.path.exists ile her iki dosyanın da diskte var olup olmadığını kontrol ediyoruz.
# "not (A and B)" -> A veya B'den herhangi biri eksikse (dosya bulunamadıysa) True olur.
if not (os.path.exists(ir_path) and os.path.exists(vis_path)):
    # Kullanıcıya, dosyaları Colab'ın sol panelindeki "Dosyalar" sekmesinden nasıl
    # yükleyebileceğini açıklayan bir uyarı mesajı yazdırıyoruz.
    print(f"Uyari: '{ir_path}' veya '{vis_path}' bulunamadi. "
          "Dosyalari sol paneldeki Dosyalar (📁) sekmesine surukleyip biraktiginizdan "
          "emin olun, sonra bu hucreyi tekrar calistirin.")
else:
    # Her iki dosya da mevcutsa, kullanıcıyı sonraki hücreye geçebileceği konusunda bilgilendiriyoruz.
    print("Dosyalar bulundu, devam edebilirsiniz.")


In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Diskteki IR ve görünür görüntü dosyalarını OpenCV ile belleğe
# okur (yükler); okuma başarısız olursa açık bir hata fırlatır ve başarılıysa
# görüntü boyutlarını (shape) yazdırarak doğrulama sağlar.
# ============================================================

# cv2.imread(ir_path, cv2.IMREAD_GRAYSCALE): IR görüntüsünü doğrudan TEK KANALLI
# (gri tonlama) olarak okur; renkli kaydedilmiş olsa bile otomatik gri tonlamaya çevirir.
ir_img = cv2.imread(ir_path, cv2.IMREAD_GRAYSCALE)
# cv2.imread(vis_path, cv2.IMREAD_COLOR): Görünür görüntüyü 3 KANALLI (BGR, renkli) olarak okur.
vis_img = cv2.imread(vis_path, cv2.IMREAD_COLOR)

# cv2.imread, dosya bulunamadığında veya bozuksa hata fırlatmak yerine None döndürür;
# bu yüzden None kontrolünü elle yapıp anlamlı bir hata mesajıyla RuntimeError fırlatıyoruz.
if ir_img is None:
    raise RuntimeError(f"IR goruntusu okunamadi: {ir_path}")
if vis_img is None:
    raise RuntimeError(f"Visible goruntusu okunamadi: {vis_path}")

# Okunan görüntülerin boyutlarını (yükseklik, genişlik[, kanal sayısı]) ekrana yazdırarak,
# dosyaların doğru şekilde ve beklenen boyutlarda yüklendiğini görsel olarak doğruluyoruz.
print("IR shape     :", ir_img.shape)
print("Visible shape:", vis_img.shape)


## 11. Füzyonu Çalıştır

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Ana füzyon pipeline'ını (fuse_maritime_images_paper) yüklenen
# IR ve görünür görüntülerle çalıştırır; ardından her önemli ara/nihai sonucun
# boyut, min, max ve ortalama istatistiklerini yazdırarak hızlı bir sağlık
# kontrolü (sanity check) sağlar.
# ============================================================

# Daha önce tanımlanan ana pipeline fonksiyonunu, okunan görüntüler ve makaledeki
# varsayılan parametrelerle (T, k, d, X, gamma, beta) çağırıyoruz.
results = fuse_maritime_images_paper(
    ir_img,
    vis_img,
    T=2,
    k=0.5,
    d=16,
    X=32,
    gamma=2.0,
    beta=0.8
)

# Pipeline'ın döndürdüğü sözlükteki en kritik ara/nihai sonuçların anahtarları üzerinde döngü kuruyoruz.
for key in ["background", "target", "ID", "Re", "IV", "IC", "fused"]:
    # results sözlüğünden ilgili array'i alıyoruz.
    arr = results[key]
    # Her array için: adı, boyutu (shape), minimum, maksimum ve ortalama değerlerini
    # hizalı (formatlanmış) şekilde yazdırıyoruz. Bu, örneğin bir ara adımın beklenmedik
    # şekilde tamamen negatif veya tamamen sabit çıkıp çıkmadığını hızlıca fark etmeyi sağlar.
    print(f"{key:12s} shape={str(arr.shape):18s} min={arr.min():8.2f} max={arr.max():8.2f} mean={arr.mean():8.2f}")


## 12. Ara Adımların Görselleştirilmesi

Bu bölüm, pipeline'ın her aşamasını ayrı ayrı gösterir. Bu sayede hangi adımın beklendiği gibi
çalıştığını (ör. arka planın pürüzsüz olması, hedefin belirginleşmesi, renklerin gerçekten geçmesi)
tek tek doğrulayabilirsiniz.

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Pipeline'ın tüm ara adımlarını (IR, arka plan, hedef, ID, görünür,
# renk alt-uzayı, ilk renkli görüntü, nihai füzyon) 2x4'lük bir ızgara (grid)
# düzeninde yan yana görselleştirir; böylece her adımın beklenen şekilde
# çalışıp çalışmadığı tek bakışta kontrol edilebilir.
# ============================================================

# 2 satır x 4 sütunluk bir subplot ızgarası oluşturuyoruz; figsize ile figürün
# toplam boyutunu (genişlik=20, yükseklik=9 inç) belirliyoruz.
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# [0,0]: Ham IR girdisini gri tonlamalı (cmap="gray") olarak gösteriyoruz.
axes[0, 0].imshow(results["ir"], cmap="gray")
axes[0, 0].set_title("IR (girdi)")

# [0,1]: Eq.7'den elde edilen arka plan görüntüsünü gösteriyoruz; vmin/vmax=0/255
# ile renk skalasının sabit (0-255) aralıkta yorumlanmasını sağlıyoruz (karşılaştırılabilirlik için).
axes[0, 1].imshow(results["background"], cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("Arka plan B(i,j) — Eq.7")

# [0,2]: Eq.12'den elde edilen hedef (target) haritasını gösteriyoruz.
axes[0, 2].imshow(results["target"], cmap="gray", vmin=0, vmax=255)
axes[0, 2].set_title("Hedef bölgesi — Eq.12")

# [0,3]: Eq.13'ten elde edilen IR degradasyon görüntüsünü (ID) gösteriyoruz.
axes[0, 3].imshow(results["ID"], cmap="gray", vmin=0, vmax=255)
axes[0, 3].set_title("IR degradasyon ID — Eq.13")

# [1,0]: Orijinal görünür görüntüyü gösteriyoruz; OpenCV BGR sırasında tuttuğu için
# Matplotlib'in beklediği RGB sırasına cv2.cvtColor ile dönüştürüyoruz.
axes[1, 0].imshow(cv2.cvtColor(results["visible"], cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("Visible (girdi)")

# [1,1]: Eq.15'ten elde edilen tek boyutlu renk alt-uzayı (Re) görüntüsünü gösteriyoruz.
axes[1, 1].imshow(results["Re"], cmap="gray")
axes[1, 1].set_title("Renk alt-uzayi R — Eq.15")

# IC görüntüsü float ve muhtemelen [0,255] dışına taşmış değerler içerebilir; imshow'a
# vermeden önce [0,255]'e kırpıp uint8'e (8-bit tam sayı) çeviriyoruz.
ic_display = np.clip(results["IC"], 0, 255).astype(np.uint8)
# [1,2]: Eq.19'dan elde edilen ilk (ham) renkli füzyon görüntüsünü RGB'ye çevirerek gösteriyoruz.
axes[1, 2].imshow(cv2.cvtColor(ic_display, cv2.COLOR_BGR2RGB))
axes[1, 2].set_title("Ilk renkli goruntu IC — Eq.19")

# [1,3]: Eq.20-22'den elde edilen NİHAİ füzyon görüntüsünü RGB'ye çevirerek gösteriyoruz.
axes[1, 3].imshow(cv2.cvtColor(results["fused"], cv2.COLOR_BGR2RGB))
axes[1, 3].set_title("Nihai fuzyon — Eq.20-22")

# İç içe döngü ile tüm alt-grafiklerin (axes) eksen çizgilerini/sayılarını (ticks) kapatıyoruz;
# görüntüler için eksen bilgisi anlamlı olmadığından temiz bir görünüm sağlar.
for ax_row in axes:
    for ax in ax_row:
        ax.axis("off")

# Alt grafikler arasındaki boşlukları otomatik olarak optimize eder (üst üste binmeyi önler).
plt.tight_layout()
# Oluşturulan figürü ekranda gösterir.
plt.show()


## 13. Nihai Karşılaştırma: IR / Görünür / Füzyon

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: IR, görünür (visible) ve nihai füzyon görüntülerini yan yana
# (1x3 ızgara) göstererek pipeline'ın genel başarısını özet şekilde karşılaştırır.
# ============================================================

# 1 satır x 3 sütunluk bir subplot ızgarası oluşturuyoruz; figür boyutu (18,6) inç.
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# [0]: Ham IR görüntüsünü gri tonlamalı gösteriyoruz.
axes[0].imshow(results["ir"], cmap="gray")
axes[0].set_title("IR")
axes[0].axis("off")  # eksen çizgilerini/sayılarını gizler

# [1]: Görünür (RGB) görüntüyü, BGR'den RGB'ye çevirerek doğru renklerle gösteriyoruz.
axes[1].imshow(cv2.cvtColor(results["visible"], cv2.COLOR_BGR2RGB))
axes[1].set_title("Visible / RGB")
axes[1].axis("off")

# [2]: Pipeline'ın ürettiği nihai füzyon görüntüsünü, yine BGR'den RGB'ye çevirerek gösteriyoruz.
axes[2].imshow(cv2.cvtColor(results["fused"], cv2.COLOR_BGR2RGB))
axes[2].set_title("Fused — Paper Pipeline (v4, duzeltilmis)")
axes[2].axis("off")

# Alt grafikler arasındaki boşlukları otomatik düzenler.
plt.tight_layout()
# Figürü ekranda gösterir.
plt.show()


## 14. Sonucu Kaydet

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Nihai füzyon görüntüsünü diske PNG dosyası olarak kaydeder.
# ============================================================

# Kaydedilecek dosyanın adını (çalışma dizinine göre göreli yol) belirliyoruz.
output_path = "fused_result_paper_method_v4.png"
# cv2.imwrite: results["fused"] (BGR formatında uint8 array) içeriğini, belirtilen
# dosya yoluna PNG formatında yazar. OpenCV varsayılan olarak BGR sırasını PNG'ye
# doğru şekilde (RGB'ye çevirerek) kaydeder, bu yüzden burada ekstra dönüşüm gerekmez.
cv2.imwrite(output_path, results["fused"])
# Kaydetme işleminin tamamlandığını ve dosya yolunu kullanıcıya bildiriyoruz.
print("Kaydedildi:", output_path)


## 15. İndir (Colab)

In [ ]:
# ============================================================
# HÜCRE İŞLEVİ: Kaydedilen füzyon sonucu PNG dosyasını, Google Colab'ın
# dosya indirme mekanizmasıyla kullanıcının bilgisayarına indirir.
# ============================================================

# google.colab.files modülü, yalnızca Colab ortamında çalışır ve tarayıcı
# üzerinden dosya indirme diyaloğu açmayı sağlar.
from google.colab import files

# Bir önceki hücrede kaydedilen output_path dosyasını kullanıcının bilgisayarına indirir.
files.download(output_path)


## 16. Sonuç Kontrolü ve Notlar

Kontrol edilmesi gereken noktalar:

- **`Re`, `o_R/o_G/o_B`**: artık `0-255` ölçeğinde olmalı (v3'te `[0,1]` mertebesinde kalıyordu — bu, renk
  bilgisinin füzyon görüntüsüne neredeyse hiç yansımamasının nedeniydi).
- **`IV`**: `ID`'nin uzamsal yapısını (arka plan + hedef bölgesi) korumalı; tamamen düz/sabit bir görüntü
  olmamalı.
- **`IC`**: hem `IV`'nin (IR kaynaklı) yapısını hem de görünür görüntünün renk tonlarını (kırmızı/mavi/yeşil
  dengesi) içermeli — griye yakın, renksiz bir görüntü OLMAMALI.
- **`fused`**: `0-255` `uint8`, IR hedefinin belirgin olduğu ve görünür görüntüye benzer renk/doku taşıyan
  bir görüntü olmalı.

Makalenin yöntemi, deniz sahnesindeki IR hedeflerini korurken görünür görüntünün renk bilgisini füzyon
görüntüsüne aktarmayı amaçlar. Makalede deneyler **hizalanmış (roughly aligned)** IR-visible çiftleri
üzerinde yapılmıştır; bu yüzden giriş görüntülerinizin de geometrik olarak birbirine yakın hizalı olması
gerekir (bu notebook yalnızca boyut eşitleme yapar, geometrik kayıt/registration yapmaz).